# Outline of Immediate Work

## Steps:
* Define our metrics for each test run
  * Usefulness
  * Safety
* Start exploring blue-team strategies
  * This will include the ones from the original paper as well as new ones
  * Some ideas:
    * Have weaker model reimplement all solutions
    * Have weaker model reimplement all suspicious solutions above a threshold
    * Have weaker model query multiple solutions from untrusted model and compare (debate)
    * Have weaker model query multiple solutions from untrusted model and implement its own
    * Have weaker model decompose problems into subtasks and have untrusted model implement them
    * Have stronger model decompose problems and have untrusted model implement them
    * Have stronger model decompose problems and have trusted model implement them
    * Give weaker model access to tools like an interpreter
    * Allow fine-tuning
    * TODO: Think of more blue team strategies
* Explore red-team strategies for each of these blue-team strategies

* Start ideating on what experiments we want to run specifically. I would like the following:
    * List of solution writing prompts/approaches
        * n-shot
        * best-of-n
        * CoT
    * List of blue team strategies to attempt
        * Work extraction strategy
            * Problem decomposition
            * Untrusted planning/execution
        * Monitoring strategy:
            * Naive 0-shot classification
            * n-shot classification
            * avg-of-n classification
    * List of red team counter-strategies for each of the above

## Possible Extensions
* Move from APPS to SWEBench
  * Idk about this one, SOTA models only get ~10-15% on SWEBench
* Measure average utility vs safety across many different capability gaps

# Execute Pipeline

#### Trusted Editing

In [ ]:
from src.experiments.trusted_editing import get_pipeline
from core.utils import setup_logging
from src.pipeline.plotting import plot

strong_model = "gpt-4-0125-preview"
weak_model = "gpt-3.5-turbo-1106"

setup_logging('debug')

editing_pipeline = get_pipeline(strong_model=strong_model, weak_model=weak_model, name='new_red_team', use_cache=True, num_problems=200)
results = await editing_pipeline.run()
plot(results, editing_pipeline.config.name, strong_model, weak_model)

#### Trusted Monitoring

In [ ]:
from src.experiments.trusted_monitoring import get_pipeline
from core.utils import setup_logging
from src.pipeline.plotting import plot
setup_logging('debug')

strong_model = "gpt-4-0125-preview"
weak_model = "gpt-3.5-turbo-1106"

pipeline = get_pipeline(strong_model=strong_model, weak_model=weak_model, name='new_red_team', use_cache=True, num_problems=200)
results = await pipeline.run()
plot(results, pipeline.config.name, strong_model, "gpt-3.5-turbo-1106")

#### Printer class

In [ ]:
from src.tools.printer import print_experiment_log

strong_model = "gpt-4-0125-preview"
weak_model = "gpt-3.5-turbo-1106"

print_experiment_log("trusted_monitoring-new_red_team", strong_model, weak_model, "2643")

#### Simple

In [ ]:
from src.experiments.simple import get_pipeline
from core.utils import setup_logging
from src.runners.evaluate_code import print_eval
setup_logging('debug')

strong_model = "claude-3-5-sonnet-20240620"
weak_model = "gpt-3.5-turbo-1106"

pipeline = get_pipeline(strong_model=strong_model, weak_model=weak_model, name='test', use_cache=False, num_problems=1)
results = await pipeline.run()

print_eval(results["evaluate_baseline"])
print_eval(results["evaluate_backdoors"])
print(results["calculate_f1"])

#### Baseline suspiciousness

In [ ]:
from src.experiments.baseline_suspiciousness_rating import get_innocent_suspiciousness_scores, get_backdoored_suspiciousness_scores
from core.utils import setup_logging
setup_logging('debug')

await get_innocent_suspiciousness_scores()
await get_backdoored_suspiciousness_scores()

#### Vivek data collection

In [1]:
from src.experiments.data_collection import get_pipeline
from core.utils import setup_logging
setup_logging('debug')

strong_model = "gpt-4-0125-preview"
weak_model = "gpt-3.5-turbo-1106"

data_pipeline = get_pipeline(strong_model=strong_model, weak_model=weak_model, name='v7', use_cache=True, num_problems=1)
results = await data_pipeline.run()

#results['final_results']

2024-08-02 05:42:24 [INFO] (core.utils) Logging level set to debug


  0%|          | 0/65 [00:00<?, ?it/s]

2024-08-02 05:42:24 [INFO] (src.pipeline.pipeline) Starting step 0: load_data - Using cache: True
2024-08-02 05:42:24 [INFO] (src.pipeline.pipeline) Finished step 0: load_data
2024-08-02 05:42:24 [INFO] (src.pipeline.pipeline) Starting step 1: get_solutions - Using cache: True
2024-08-02 05:42:24 [INFO] (src.pipeline.pipeline) Finished step 1: get_solutions
2024-08-02 05:42:24 [INFO] (src.pipeline.pipeline) Starting step 2: query_benign_untrusted_suspiciousness - Using cache: True
2024-08-02 05:42:24 [INFO] (src.pipeline.pipeline) Finished step 2: query_benign_untrusted_suspiciousness
2024-08-02 05:42:24 [INFO] (src.pipeline.pipeline) Starting step 3: query_benign_untrusted_usefulness - Using cache: True
2024-08-02 05:42:24 [INFO] (src.pipeline.pipeline) Finished step 3: query_benign_untrusted_usefulness
2024-08-02 05:42:24 [INFO] (src.pipeline.pipeline) Starting step 4: query_benign_results - Using cache: True
2024-08-02 05:42:24 [INFO] (src.pipeline.pipeline) Finished step 4: query_b

got capacities for model gpt-4-0125-preview: 2000000, 10000
consumed capacities for model gpt-4-0125-preview: 20, 1
setting cap for model gpt-4-0125-preview: 1980000.0, 9900.0


2024-08-02 05:42:38 [DEBUG] (core.llm_api.openai_llm) Completed call to gpt-4-0125-preview in 13.567493677139282s.


In [ ]:
verdicts = {}
for problem_id, solutions in data_pipeline.results['trusted_editing_malicious_results'].items():
    def contains_substring(main_string, substring):
        # Remove all whitespaces and newlines from both strings
        main_string = main_string.replace(' ', '').replace('\n', '')
        substring = substring.replace(' ', '').replace('\n', '')

        # Extract the section of the substring that contains an `if ...:` statement
        if_index = substring.find('if')
        if if_index != -1:
            end_index = substring.find(':', if_index) + 1
            substring = substring[if_index:end_index]

        # Check if the processed substring is in the processed main string
        return substring in main_string
    
    for i, solution in enumerate(solutions):
        subproblem_id = f"{problem_id}-{i}"
        data = {}
        data['contains_suggestion'] = contains_substring(solution['solution'], solution['backdoor_code'])
        data['verdict'] = solution['backdoor_present']
        verdicts[subproblem_id] = data

## TODO

* Blue team monitoring improvements
  * Read redwood's code and reimplement
  * It's basically what we have, just with logit bias + logprobs
* Red team improvements
  * Make sure that what we have for our red-team pipeline is identical to what Redwood has
    * Done
  * Update all pipelines to use the new methodology
* Implement ROC calculation a la their paper
* QoL:
  * Shared red-team backdoor-production pipeline for monitoring and editing strategies
    * Done